In [ ]:
from infostop import Infostop
import numpy as np
import polars as pl

import yaml

In [2]:
with open("05_detect_stops_config.yaml") as fp:
    config = yaml.safe_load(fp)


In [3]:
def initial_filter(df: pl.DataFrame, min_pings: int) -> pl.DataFrame:
    """Keep only devices with at least min_pings pings."""
    valid_devices = (
        df.group_by("device_id")
        .len()
        .filter(pl.col("len") >= min_pings)["device_id"]
        .to_list()
    )
    return df.filter(pl.col("device_id").is_in(valid_devices))


In [4]:
data = pl.read_parquet(
    "/mnt/common-hdd/group-dirs/tkom-2026-project/partitioned/00/pings.parquet"
)


In [5]:
filtered = initial_filter(data, min_pings=config["filtering"]["min_pings_per_day"])

In [6]:
filtered = filtered.with_columns(
    pl.col("timestamp").dt.epoch(time_unit="s").alias("epoch")
)

In [7]:
# stop detection process
params = config["infostop"]

model = Infostop(
    r1=params["r1"],
    r2=params["r2"],
    min_staying_time=params["min_staying_time"],
    max_time_between=params["max_time_between"],
    min_size=params["min_size"],
    distance_metric=params["distance_metric"],
)

In [8]:
# run infostop per device — data already sorted, no re-sort needed in loop
# catch devices with no stop events (all pings labelled -1)
results = []
for device_id, group in filtered.group_by("device_id"):
    coords = group[["latitude", "longitude", "epoch"]].to_numpy()
    try:
        labels = model.fit_predict(coords)
    except Exception:
        labels = np.full(len(coords), -1)
    group_ = group.with_columns(stop_label=labels)

    results.append(group_)

stops = pl.concat(results)
stops = stops.sort(["device_id", "timestamp"])

n_stopped = (stops["stop_label"] >= 0).sum()
print(f"Stop pings:    {n_stopped:,}  ({n_stopped / len(stops):.1%} of filtered pings)")
print(f"Unique stops:  {stops['stop_label'].clip(lower_bound=0).n_unique():,}")
stops.head(3)


Stop pings:    9,474,653  (61.3% of filtered pings)
Unique stops:  393


timestamp,device_id,device_aid_type,latitude,longitude,epoch,stop_label
"datetime[μs, Europe/Budapest]",i64,str,f64,f64,i64,i64
2024-10-02 02:37:09 CEST,1,"""AAID""",47.47992,19.074155,1727829429,2
2024-10-02 03:17:49 CEST,1,"""AAID""",47.479995,19.074221,1727831869,2
2024-10-02 03:26:50 CEST,1,"""AAID""",47.476752,19.075154,1727832410,-1


In [9]:
stops.drop("epoch").write_parquet(
    "/mnt/common-hdd/group-dirs/tkom-2026-project/stops/00.parquet"
)